# Silver Calendar Dimension Table

Creates a comprehensive date dimension table `mano_adb_outlook.silver.Calendar` spanning 1940–2030 with 36 analytical fields including fiscal year, relative date flags, and period boundaries.

In [0]:
from pyspark.sql.functions import *

# Generate date range: 1940-01-01 to 2030-12-31
df_calendar = (
    spark.sql("SELECT explode(sequence(DATE '1940-01-01', DATE '2030-12-31', INTERVAL 1 DAY)) AS date")
    .withColumn("date_key", date_format("date", "yyyyMMdd").cast("int"))
    .withColumn("day", dayofmonth("date"))
    .withColumn("day_name", date_format("date", "EEEE"))
    .withColumn("day_short", date_format("date", "EEE"))
    .withColumn("day_of_week", dayofweek("date"))            # 1=Sun, 7=Sat
    .withColumn("day_of_week_iso",
        when(dayofweek("date") == 1, 7).otherwise(dayofweek("date") - 1))  # 1=Mon, 7=Sun
    .withColumn("day_of_year", dayofyear("date"))
    .withColumn("week_of_year", weekofyear("date"))
    .withColumn("month", month("date"))
    .withColumn("month_name", date_format("date", "MMMM"))
    .withColumn("month_short", date_format("date", "MMM"))
    .withColumn("quarter", quarter("date"))
    .withColumn("quarter_name", concat(lit("Q"), quarter("date").cast("string")))
    .withColumn("year", year("date"))
    .withColumn("year_month", date_format("date", "yyyy-MM"))
    .withColumn("year_quarter", concat(year("date").cast("string"), lit("-Q"), quarter("date").cast("string")))
    .withColumn("is_weekend", dayofweek("date").isin(1, 7).cast("boolean"))
    .withColumn("is_weekday", (~dayofweek("date").isin(1, 7)).cast("boolean"))
    .withColumn("is_leap_year",
        (((year("date") % 4 == 0) & (year("date") % 100 != 0)) | (year("date") % 400 == 0)).cast("boolean"))
    .withColumn("days_in_month", dayofmonth(last_day("date")))
    .withColumn("week_start_monday", date_sub("date", (dayofweek("date") + 5) % 7))
    .withColumn("week_end_sunday", date_add(date_sub("date", (dayofweek("date") + 5) % 7), 6))
    .withColumn("start_of_month", trunc("date", "month"))
    .withColumn("end_of_month", last_day("date"))
    .withColumn("start_of_quarter", trunc("date", "quarter"))
    .withColumn("end_of_quarter", last_day(add_months(trunc("date", "quarter"), 2)))
    .withColumn("start_of_year", trunc("date", "year"))
    .withColumn("end_of_year", make_date(year("date"), lit(12), lit(31)))
    # Fiscal year (July start)
    .withColumn("fiscal_year",
        when(month("date") >= 7, year("date") + 1).otherwise(year("date")))
    .withColumn("fiscal_quarter",
        when(month("date").between(7, 9), lit(1))
        .when(month("date").between(10, 12), lit(2))
        .when(month("date").between(1, 3), lit(3))
        .otherwise(lit(4)))
    .withColumn("fiscal_month",
        when(month("date") >= 7, month("date") - 6).otherwise(month("date") + 6))
    # Relative flags (useful for dashboards)
    .withColumn("is_current_day", (col("date") == current_date()).cast("boolean"))
    .withColumn("is_last_30_days",
        ((col("date") >= date_sub(current_date(), 30)) & (col("date") <= current_date())).cast("boolean"))
    .withColumn("is_last_90_days",
        ((col("date") >= date_sub(current_date(), 90)) & (col("date") <= current_date())).cast("boolean"))
    .withColumn("is_ytd",
        ((col("date") >= trunc(current_date(), "year")) & (col("date") <= current_date())).cast("boolean"))
)

# Write to silver as a managed Delta table
df_calendar.write.mode("overwrite").saveAsTable("mano_adb_outlook.silver.Calendar")

print(f"Calendar table created with {df_calendar.count()} rows")
print(f"Date range: 1940-01-01 to 2030-12-31")
display(df_calendar.orderBy("date").limit(10))